<div style="text-align: center;"><img src="https://www.databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" width="600"></div>


# Machine Learning Pipeline with Databricks API

In this demo, we configure and initiate a Databricks Lakeflow Job to run a series of tasks representing an MLOps pipeline. Using the Databricks REST API, we will create, execute, and monitor a series of four notebooks in sequence.

**Learning Objectives**

By the end of this demo, you will be able to:

- **Authentication Setup**
  - Configure and authenticate access to the Databricks REST API.

- **Pipeline Configuration**
  - Define and initialize a JSON payload for pipeline tasks.

- **Executing the Pipeline**
  - Trigger the Job using the Databricks REST API.

- **Monitoring Task Progress**
  - Track job status and task completion using REST API calls.

- **Notifications on Completion**
  - Set up email notifications for Job completion.

- **Retrieving and Displaying Outputs**
  - Access and visualize JSON data and output files from completed tasks.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong> — <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo.

**Important:** Ensure that you have completed the [**`0_Generate_Tokens`**]($./0_Generate_Tokens) notebook beforehand. This will set up the necessary authentication credentials.

**If you have not done so, complete the [**`0_Generate_Tokens`**]($./0_Generate_Tokens) notebook now, then start this notebook again from the beginning to ensure proper setup.**

Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-2.1

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

<div style="width: 100%; font-family: sans-serif;"><div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">  <img src="https://files.training.databricks.com/binder/prod_main/machine-learning-operations-en_us-2.4.2/images/20261111T155619Z/Machine Learning Operations/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">  <div style="font-size: 15pt; color: #0B2026; line-height: 1.7; margin-bottom: 16px;">    Want to know more about building machine learning pipelines with the Databricks REST API? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/machine-learning-operations-en_us-2.4.2/images/20261111T155619Z/Machine Learning Operations/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.  </div>  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0B2026;">    <span id="genie-query-2-1" style="flex: 1;">How can I use the Databricks REST API (Jobs API) to programmatically create and run a multi-task machine learning pipeline for data ingestion, transformation, and feature engineering? How do I pass values between tasks and retrieve a task's output?</span>    <button onclick="      var text = document.getElementById('genie-query-2-1').innerText;      var ta = document.createElement('textarea');      ta.value = text;      ta.style.position = 'fixed';      ta.style.opacity = '0';      document.body.appendChild(ta);      ta.select();      document.execCommand('copy');      document.body.removeChild(ta);      this.innerText = 'Copied!';      var btn = this;      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>  </div></div></div>

## Pipeline Configuration and Setup
In this section, we set up the environment and define paths for each notebook in the pipeline. We also configure the API headers for communication with the Databricks REST API.

**Instructions:**

Here, we initialize the foundational configuration by defining:

- **Databricks Instance URL:** Constructs the instance URL using the workspace configuration.
- **API Token:** Defines the token for authorization, which is essential for authenticated API requests.
- **API Headers:** Sets the headers for the API requests, including the authorization token and content type to ensure secure and correctly formatted requests.

In [0]:
import requests
import json

# Databricks instance and API token
databricks_instance = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
api_token = token.strip("{}").strip("'")

# Define headers for API requests
headers = {
    "Authorization": f"Bearer {api_token}",
    "Content-Type": "application/json"
}

### Define Notebook Paths for Pipeline Tasks
In this section, we dynamically determine the paths to each notebook involved in the pipeline. This setup ensures all task notebooks are linked correctly for sequential execution.

- **Current Notebook Path:** Retrieves the path of the current notebook.
- **Base Path:** Defines a base path for locating all related notebooks in the job pipeline.
- **Task Notebook Paths:** Specifies the paths for each task notebook (Data Ingestion, Data Transformation, Advanced Feature Engineering, and Displaying Feature Engineered Data) to ensure they’re accessible within the pipeline.

In [0]:
# Get the current notebook path
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Base path for related notebooks
base_path = notebook_path.rsplit('/', 1)[0] + "/2.3_Demo_Pipeline_Data_Preparation"

# Paths for specific process notebooks
notebook1 = f"{base_path}/2.3a_Data_Ingestion"
print(notebook1)
notebook2 = f"{base_path}/2.3b_Data_Transformation" 
print(notebook2)
notebook3 = f"{base_path}/2.3c_Advanced_Feature_Engineering_(Build_Features)" 
print(notebook3)
notebook4 = f"{base_path}/2.3d_Displaying_Feature_Engineered_Data"
print(notebook4)
# Store the notebook paths in a list for easy iteration
notebook_paths = [notebook1, notebook2, notebook3, notebook4]

##Pipeline Tasks Creation

In this section, we use the `Databricks REST API` to define tasks for each notebook in the pipeline. Each task is dependent on the previous one, ensuring sequential execution. An email notification is sent upon successful job completion.

**Instructions:**

The **`create_job_with_tasks`** function allows for a modular approach to building Jobs. Each task is specified by a unique **`task_key`**, and dependencies are established between tasks. 

> ****Important:**** Email notifications ensure that you are alerted as soon as the job completes.

In [0]:
# Function to create a job with tasks, including email notifications for success
def create_job_with_tasks(job_name, notebook_paths):
    tasks = []
    for i, notebook_path in enumerate(notebook_paths):
        task = {
            "task_key": f"task_{i+1}",
            "notebook_task": {
                "notebook_path": notebook_path
            }
        }
        
        # Add dependencies for each subsequent task
        if i > 0:
            task["depends_on"] = [{"task_key": f"task_{i}"}]
        
        tasks.append(task)
    
    # Job payload with email notifications
    job_payload = {
        "name": f"{DA.username}-testing-workflow-job",
        "tasks": tasks,
        "email_notifications": {
            "on_success": [DA.username],  # Sends an email notification on success to the user
            "no_alert_for_skipped_runs": False
        },
        "notification_settings": {
            "no_alert_for_skipped_runs": False,
            "no_alert_for_canceled_runs": False
        }
    }
    
    # Create the job using the Databricks REST API
    response = requests.post(
        f"{databricks_instance}/api/2.1/jobs/create",
        headers=headers,
        data=json.dumps(job_payload)
    )
    
    if response.status_code == 200:
        job_id = response.json().get("job_id")
        print(f"Job '{job_name}' created with ID: {job_id}")
        return job_id
    else:
        print(f"Error creating job: {response.text}")
        return None

# Define the job name and create the job with tasks
job_name = "ML Pipeline Workflow"
job_id = create_job_with_tasks(job_name, notebook_paths)

##Running the Pipeline Job
This section triggers the Lakeflow Job after all tasks are defined, initiating the pipeline's execution.

**Instructions:**

The `run_job` function uses the Databricks REST API to start the Lakeflow Job by specifying the `job_id`. The output `run_id` is important for tracking this specific job execution in future requests. Use this function to automate the process and ensure timely task execution.

In [0]:
def run_job(job_id):
    # Send a POST request to start the job with the given job_id
    response = requests.post(
        f"{databricks_instance}/api/2.1/jobs/run-now",
        headers=headers,
        data=json.dumps({"job_id": job_id})
    )
    
    # Check if the request was successful
    if response.status_code == 200:
        # Extract the run_id from the response
        run_id = response.json().get("run_id")
        print(f"Job '{job_name}' started with Run ID: {run_id}")
        return run_id
    else:
        # Print the error message if the request failed
        print(f"Error starting job: {response.text}")
        return None

# Start the job and get the run_id
run_id = run_job(job_id)

### Accessing the Jobs
After the job is started, you can access the job details and log history directly from the Databricks Jobs interface. This final cell provides a direct link to the Lakeflow Job, allowing instructors or users to review task execution details.

**Instruction:**

The link generated is a direct access point to monitor the job execution and review logs or any error messages. This link is valuable for troubleshooting or assessing job performance metrics.

In [0]:
# Print the link to the Jobs
workspace_id = databricks_instance.split("//")[1].split(".")[0]
print(f"Job started successfully. You can monitor the progress at: https://{databricks_instance.split('//')[1]}/jobs/{job_id}?o={workspace_id}")

##Retrieving Task Outputs and Visualization
This function polls the job status to ensure it completes successfully, then retrieves the JSON data output and visualizes the PNG file generated in the final notebook.

**Instructions:**

The `get_final_task_output` function checks the job's state and result status, displaying the **JSON** data and visualization **PNG** file. It also sends an email upon job completion, useful for monitoring and collaboration.

In [0]:
# Function to wait for the job to complete and retrieve the output from JSON and PNG files
import time
import os
def get_final_task_output(run_id, 
                          json_output_path="2.3_Demo_Pipeline_Data_Preparation/feature_engineered_output_with_visualization_data.json", 
                          png_output_path="2.3_Demo_Pipeline_Data_Preparation/total_assets_by_credit_usage_category.png"):
    from PIL import Image
    import matplotlib.pyplot as plt

    # Polling to check job status
    while True:
        response = requests.get(
            f"{databricks_instance}/api/2.1/jobs/runs/get",
            headers=headers,
            params={"run_id": run_id}
        )
        
        if response.status_code == 200:
            job_status = response.json().get("state", {}).get("life_cycle_state")
            result_state = response.json().get("state", {}).get("result_state")
            print(f"Job status: {job_status}, Result state: {result_state}")
            
            if job_status == "TERMINATED":
                if result_state == "SUCCESS":
                    # Attempt to read the JSON output file once the job is successful
                    try:
                        # Display JSON file contents
                        with open(json_output_path, 'r') as file:
                            task_output = json.load(file)
                            print("\n=======\nOutput of Final Task (JSON Data):\n")
                            print(json.dumps(task_output, indent=4))
                            
                        # Confirm the existence of the PNG image file
                        if os.path.exists(png_output_path):
                            # Display the bar graph from the PNG file
                            try:
                                img = Image.open(png_output_path)
                                plt.imshow(img)
                                plt.axis('off')
                                plt.show()
                            except Exception as e:
                                print(f"Error displaying the bar graph: {e}")
                            print(f"\nVisualization image saved at: {png_output_path}")
                        else:
                            print("Visualization image file not found.")
                    except Exception as e:
                        print(f"Error reading final task output: {e}")
                else:
                    print("Job did not complete successfully.")
                break
        else:
            print(f"Error getting job status: {response.text}")
            break

        time.sleep(20)  # Wait before polling again

# Retrieve and print the output of the final task
get_final_task_output(run_id)

##Conclusion
In this notebook, we demonstrated how to configure and execute a Databricks Lakeflow Job to implement an MLOps pipeline. Using Databricks' REST API, we created a sequence of tasks to handle data ingestion, transformation, feature engineering, and data visualization. The workflow was automated to run each notebook in sequence, and a monitoring function provided real-time updates, including a visualization output and email notifications upon completion.

This pipeline represents a scalable approach to managing machine learning workflows, aligning with the MLOps principles.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
